In [2]:
import pandas as pd
import json
import os

with open(os.path.expanduser('~/projects/statsbomb-data/data/competitions.json')) as f:
    data = json.load(f)

df = pd.DataFrame(data)

target_leagues = ["Premier League", "La Liga", "Serie A", "Ligue 1"]
target_season = "2015/2016"

selected = df[(df['competition_name'].isin(target_leagues)) & (df['season_name'] == target_season)]
print(selected[['competition_name', 'competition_id', 'season_id', 'season_name']])

   competition_name  competition_id  season_id season_name
45          La Liga              11         27   2015/2016
63          Ligue 1               7         27   2015/2016
68   Premier League               2         27   2015/2016
70          Serie A              12         27   2015/2016


In [3]:
import json, os
path = os.path.expanduser('~/projects/statsbomb-data/data/matches/11/27.json')
with open(path) as f:
    matches = json.load(f)
print(matches[0])

print()

match_ids = []
for match in matches:
    match_ids.append(match['match_id'])

print(len(match_ids))

{'match_id': 3825739, 'match_date': '2016-01-17', 'kick_off': '17:00:00.000', 'competition': {'competition_id': 11, 'country_name': 'Spain', 'competition_name': 'La Liga'}, 'season': {'season_id': 27, 'season_name': '2015/2016'}, 'home_team': {'home_team_id': 220, 'home_team_name': 'Real Madrid', 'home_team_gender': 'male', 'home_team_group': None, 'country': {'id': 214, 'name': 'Spain'}, 'managers': [{'id': 56, 'name': 'Zinédine Zidane', 'nickname': None, 'dob': '1972-06-23', 'country': {'id': 78, 'name': 'France'}}]}, 'away_team': {'away_team_id': 1041, 'away_team_name': 'Sporting Gijón', 'away_team_gender': 'male', 'away_team_group': None, 'country': {'id': 214, 'name': 'Spain'}, 'managers': [{'id': 187, 'name': 'Abelardo Fernández Antuña', 'nickname': 'Abelardo', 'dob': '1970-04-19', 'country': {'id': 214, 'name': 'Spain'}}]}, 'home_score': 5, 'away_score': 1, 'match_status': 'available', 'match_status_360': 'unscheduled', 'last_updated': '2024-05-16T14:06:52.149840', 'last_updated

In [4]:
match_ids = []

for idx, row in selected.iterrows():
    comp_id = row['competition_id']
    season_id = row['season_id']
    path = os.path.expanduser(f'~/projects/statsbomb-data/data/matches/{comp_id}/{season_id}.json')
    with open(path) as f:
        matches = json.load(f)
    for match in matches:
        match_ids.append(match['match_id'])

print(len(match_ids))

1517


In [5]:
print(len(match_ids))

1517


In [6]:
import duckdb
import os
import json
import pandas as pd

def get_match_events(match_id):
    path = os.path.expanduser(f"~/projects/statsbomb-data/data/events/{match_id}.json")
    query = f"""
    SELECT
        {match_id} AS match_id,
        id,
        index,
        minute,
        second,
        timestamp,
        type.name AS type,
        possession,
        play_pattern.name AS play_pattern,
        team.name AS team,
        player.id AS player_id,
        player.name AS player_name,
        position.name AS position,
        location[1] AS x,
        location[2] AS y,
        duration,
        under_pressure
    FROM read_json_auto('{path}')
    ORDER BY index
    """
    return duckdb.sql(query).df()


def get_minutes_played(match_id):
    path = os.path.expanduser(f"~/projects/statsbomb-data/data/events/{match_id}.json")

    with open(path) as f:
        raw_events = json.load(f)

    starting_xi_events = [e for e in raw_events if e['type']['name'] == 'Starting XI']
    sub_events = [e for e in raw_events if e['type']['name'] == 'Substitution']

    start_minutes = {}
    player_names = {}

    for lineup_event in starting_xi_events:
        for player_entry in lineup_event['tactics']['lineup']:
            pid = player_entry['player']['id']
            start_minutes[pid] = 0
            player_names[pid] = player_entry['player']['name']

    end_minutes = {}
    for sub in sub_events:
        off_id = sub['player']['id']
        on_id = sub['substitution']['replacement']['id']
        end_minutes[off_id] = sub['minute']
        start_minutes[on_id] = sub['minute']
        player_names[on_id] = sub['substitution']['replacement']['name']

    final_minute = max(e['minute'] for e in raw_events)

    minutes_played = {}
    for pid in start_minutes:
        player_end = end_minutes.get(pid, final_minute)
        minutes_played[pid] = player_end - start_minutes[pid]

    df = pd.DataFrame.from_dict(minutes_played, orient='index')
    df = df.reset_index()
    df.columns = ['player_id', 'minutes_played']
    df['player_name'] = df['player_id'].map(player_names)
    df['match_id'] = match_id
    return df

print(get_match_events(3754217).head())

   match_id                                    id  index  minute  second  \
0   3754217  9d86a178-3514-45d1-9d14-1372e846d17b      1       0       0   
1   3754217  82acc213-90f3-4fee-8305-bd9e403cec42      2       0       0   
2   3754217  d18f1a33-65ba-42e5-a9fd-bbfc709694e8      3       0       0   
3   3754217  49147091-455c-4372-8d56-96e1ca71d01a      4       0       0   
4   3754217  5f9bdb6d-0379-4f42-9f53-29777f166db5      5       0       0   

         timestamp         type  possession   play_pattern     team  \
0         00:00:00  Starting XI           1   Regular Play  Chelsea   
1         00:00:00  Starting XI           1   Regular Play  Arsenal   
2         00:00:00   Half Start           1   Regular Play  Chelsea   
3         00:00:00   Half Start           1   Regular Play  Arsenal   
4  00:00:00.622000         Pass           2  From Kick Off  Arsenal   

   player_id   player_name        position     x     y  duration  \
0       <NA>           NaN             NaN   NaN

In [7]:
all_events = []
all_minutes = []
failed_matches = []

for i, mid in enumerate(match_ids):
    try:
        all_events.append(get_match_events(mid))
        all_minutes.append(get_minutes_played(mid))
    except Exception as e:
        failed_matches.append((mid, str(e)))
    if (i + 1) % 100 == 0:
        print(f"Processed {i+1}/{len(match_ids)}")

print(f"Failed: {len(failed_matches)}")
print(failed_matches[:5])

Processed 100/1517
Processed 200/1517
Processed 300/1517
Processed 400/1517
Processed 500/1517
Processed 600/1517
Processed 700/1517
Processed 800/1517
Processed 900/1517
Processed 1000/1517
Processed 1100/1517
Processed 1200/1517
Processed 1300/1517
Processed 1400/1517
Processed 1500/1517
Failed: 0
[]


In [13]:
events_df = pd.concat(all_events, ignore_index=True)
minutes_df = pd.concat(all_minutes, ignore_index=True)

print(events_df.shape)
print(minutes_df.shape)

print(events_df)
print(minutes_df)

(5321459, 17)
(42096, 4)
         match_id                                    id  index  minute  \
0         3825739  05140693-8f70-4f71-b461-fbf87043a0b8      1       0   
1         3825739  e0bdadc6-8e1e-4aac-a270-5e2652e3a42b      2       0   
2         3825739  db8bd6d9-401f-4d5b-a2cd-ee4eaafb8515      3       0   
3         3825739  28aa5950-9d12-4a29-a9f9-1637c17569c4      4       0   
4         3825739  a7bc8dba-ace5-4608-bfa0-ce8e6d24f17e      5       0   
...           ...                                   ...    ...     ...   
5321454   3878540  7c4c3eae-84c0-4da0-baa4-e87d11376565   3889      93   
5321455   3878540  d4d557ff-725e-4cfa-afee-e14fa9e3ed4b   3890      93   
5321456   3878540  5c115da5-a558-44c9-a0a9-b229101fb142   3891      94   
5321457   3878540  c2c7713a-43b2-4174-8fa7-dbbc83ce50a3   3892      94   
5321458   3878540  3445ba63-45ab-4c82-b73a-4f75822c03fb   3893      94   

         second        timestamp           type  possession   play_pattern  \
0       

In [ ]:
con = duckdb.connect('../your_database.duckdb')

con.execute("CREATE OR REPLACE TABLE events AS SELECT * FROM events_df")
con.execute("CREATE OR REPLACE TABLE player_minutes AS SELECT * FROM minutes_df")

print(con.execute("SHOW TABLES").fetchall())

In [15]:
print(con.execute("SELECT COUNT(DISTINCT player_id) FROM player_minutes").fetchone())
print(con.execute("SELECT COUNT(DISTINCT match_id) FROM player_minutes").fetchone())
print(con.execute("SELECT player_name, SUM(minutes_played) as total FROM player_minutes GROUP BY player_id, player_name ORDER BY total DESC LIMIT 5").fetchall())

(2176,)
(1517,)
[('Kasper Schmeichel', 3581), ('Wes Morgan', 3581), ('Andrew Surman', 3578), ('Simon Francis', 3578), ('Toby Alderweireld', 3558)]


In [16]:
gold_query = """
CREATE OR REPLACE TABLE player_season AS
SELECT
    m.player_id,
    m.player_name,
    SUM(m.minutes_played) AS total_minutes,
    COUNT(DISTINCT m.match_id) AS matches_played,
    COUNT(CASE WHEN e.type = 'Shot' THEN 1 END) AS total_shots,
    COUNT(CASE WHEN e.type = 'Pass' THEN 1 END) AS total_passes
FROM player_minutes m
LEFT JOIN events e
    ON m.player_id = e.player_id AND m.match_id = e.match_id
GROUP BY m.player_id, m.player_name
"""

con.execute(gold_query)
print(con.execute("SELECT * FROM player_season ORDER BY total_minutes DESC LIMIT 5").fetchdf())

   player_id                player_name  total_minutes  matches_played  \
0       7024    Jorge Luiz Frello Filho      1017835.0              35   
1       4506             Nampalys Mendy       880941.0              38   
2       7025               Marek Hamšík       875959.0              38   
3       3359          Jean Michaël Seri       838748.0              38   
4       3478  Francesc Fàbregas i Soler       831602.0              37   

   total_shots  total_passes  
0            6          3779  
1            6          2991  
2           84          3094  
3           51          2811  
4           48          2947  


In [17]:
gold_query = """
CREATE OR REPLACE TABLE player_season AS
WITH event_agg AS (
    SELECT
        player_id,
        match_id,
        COUNT(CASE WHEN type = 'Shot' THEN 1 END) AS shots,
        COUNT(CASE WHEN type = 'Pass' THEN 1 END) AS passes
    FROM events
    GROUP BY player_id, match_id
)
SELECT
    m.player_id,
    m.player_name,
    SUM(m.minutes_played) AS total_minutes,
    COUNT(DISTINCT m.match_id) AS matches_played,
    SUM(COALESCE(e.shots, 0)) AS total_shots,
    SUM(COALESCE(e.passes, 0)) AS total_passes
FROM player_minutes m
LEFT JOIN event_agg e
    ON m.player_id = e.player_id AND m.match_id = e.match_id
GROUP BY m.player_id, m.player_name
"""

con.execute(gold_query)
print(con.execute("SELECT * FROM player_season ORDER BY total_minutes DESC LIMIT 5").fetchdf())

   player_id        player_name  total_minutes  matches_played  total_shots  \
0       3815  Kasper Schmeichel         3581.0              38          0.0   
1       3813         Wes Morgan         3581.0              38         21.0   
2       3608      Simon Francis         3578.0              38          6.0   
3       3344      Andrew Surman         3578.0              38         16.0   
4      20005  Toby Alderweireld         3558.0              38         41.0   

   total_passes  
0        1220.0  
1         849.0  
2        2454.0  
3        2309.0  
4        2122.0  
